# PCA → $k$-Means Pipeline

---

## Overview

A common pipeline for high-dimensional clustering:

1. **PCA** reduces dimensions while preserving variance. removes noise and speeds up clustering
2. **$k$-Means** clusters the low-dimensional representation

This approach is especially useful when the dataset has many correlated features.

## Why PCA First?

- Euclidean distance in high dimensions is dominated by noise (curse of dimensionality)
- PCA removes redundant directions, making distance metrics more meaningful
- Reduces computational cost of $k$-Means assignment step from $O(Nkd)$ to $O(Nkk')$ where $k' \ll d$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import PCA, KMeans
from rice_ml.preprocess import StandardScaler

In [ ]:
try:
    df = pd.read_csv('../../../data/gym_members_exercise_tracking.csv')
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    X = df[num_cols].dropna().values.astype(float)
    print(f'Loaded gym dataset: {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import load_iris
    iris = load_iris()
    X = iris.data
    print('CSV not found. using iris dataset')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Shape: {X_scaled.shape}')

## Step 1. PCA: Choose number of components

In [ ]:
pca_full = PCA(n_components=X_scaled.shape[1])
pca_full.fit(X_scaled)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)

n_components = int(np.argmax(cumulative >= 0.90)) + 1
print(f'Components for 90% variance: {n_components}')

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative) + 1), cumulative * 100, marker='o', color='steelblue')
plt.axhline(90, color='red', linestyle='--', label='90% threshold')
plt.axvline(n_components, color='salmon', linestyle=':', label=f'k={n_components}')
plt.xlabel('Number of Components', fontsize=15)
plt.ylabel('Cumulative Variance (%)', fontsize=15)
plt.title('PCA: Cumulative Explained Variance', fontsize=18)
plt.legend(fontsize=13)
plt.show()

In [ ]:
# Apply PCA reduction
pca = PCA(n_components=n_components)
X_pca = pca.fit_transform(X_scaled)
print(f'Reduced shape: {X_pca.shape}')

## Step 2. $k$-Means on PCA-reduced data

Use the elbow method on the reduced data to find the best $k$.

In [ ]:
inertias_raw = []
inertias_pca = []
k_range = range(1, 8)

for k in k_range:
    km_raw = KMeans(k=k, max_iters=100)
    km_raw.fit(X_scaled)
    inertias_raw.append(km_raw.inertia_)

    km_pca = KMeans(k=k, max_iters=100)
    km_pca.fit(X_pca)
    inertias_pca.append(km_pca.inertia_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
ax1.plot(k_range, inertias_raw, marker='o', color='steelblue')
ax1.set_title('Elbow: Raw Features', fontsize=14)
ax1.set_xlabel('k'); ax1.set_ylabel('Inertia')

ax2.plot(k_range, inertias_pca, marker='o', color='salmon')
ax2.set_title(f'Elbow: PCA ({n_components} components)', fontsize=14)
ax2.set_xlabel('k'); ax2.set_ylabel('Inertia')

plt.suptitle('Elbow Method: Raw vs PCA Features', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
k_best = 3
km = KMeans(k=k_best, max_iters=200)
km.fit(X_pca)

# Project to 2D for visualization
X_2d = PCA(n_components=2).fit_transform(X_scaled)
colors = ['red', 'lightseagreen', 'steelblue', 'magenta', 'orange']

plt.figure(figsize=(10, 8))
for j in range(k_best):
    mask = km.labels_ == j
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=colors[j], label=f'Cluster {j}', alpha=0.7)

plt.xlabel('PC 1', fontsize=15)
plt.ylabel('PC 2', fontsize=15)
plt.title(f'PCA → $k$-Means (k={k_best})', fontsize=18)
plt.legend(fontsize=13)
plt.show()

## Interpretation

- PCA denoises the data before clustering, often producing **cleaner cluster boundaries**.
- The inertia on PCA-reduced data is not directly comparable to raw-feature inertia (different scales), but the **elbow shape** is often more pronounced.
- This pipeline is standard in practice: PCA first, then any distance-based algorithm.